In [1]:
class AMeta(type):
    def __new__(mcs, name, bases, classdict, **kwargs):
        print(f"AMeta.__new__ {name=}, {bases=}, {classdict=}, {kwargs=}")

        cls = super().__new__(mcs, name, bases, classdict)
        print("AMeta __new__", cls)

        return cls

    def __init__(cls, name, bases, classdict, **kwargs):
        print("AMeta __init__", cls, name, bases, classdict)
        super().__init__(name, bases, classdict, **kwargs)

    def __call__(cls, *args, **kwargs):
        print("AMeta __call__", cls, args, kwargs)
        return super().__call__(*args, **kwargs)

    @classmethod
    def __prepare__(mcs, name, bases, **kwargs):
        print("AMeta __prepare__", mcs, name, bases, kwargs)
        
        return {"added_attr_a": 2, "added_attr_b": 99}


class Person(metaclass=AMeta):
    age = 99

    def __new__(cls, *args, **kwargs):
        print("Person __new__", cls, *args, **kwargs)
        return super().__new__(cls)

    def __init__(self, name):
        print("Person __init__", name)
        self.name = name

AMeta __prepare__ <class '__main__.AMeta'> Person () {}
AMeta.__new__ name='Person', bases=(), classdict={'added_attr_a': 2, 'added_attr_b': 99, '__module__': '__main__', '__qualname__': 'Person', '__firstlineno__': 25, 'age': 99, '__new__': <function Person.__new__ at 0x10ca10eb0>, '__init__': <function Person.__init__ at 0x10ca10f60>, '__static_attributes__': ('name',), '__classdictcell__': <cell at 0x10c9a6440: dict object at 0x10ca27000>, '__classcell__': <cell at 0x10c9a6770: empty>}, kwargs={}
AMeta __new__ <class '__main__.Person'>
AMeta __init__ <class '__main__.Person'> Person () {'added_attr_a': 2, 'added_attr_b': 99, '__module__': '__main__', '__qualname__': 'Person', '__firstlineno__': 25, 'age': 99, '__new__': <function Person.__new__ at 0x10ca10eb0>, '__init__': <function Person.__init__ at 0x10ca10f60>, '__static_attributes__': ('name',), '__classdictcell__': <cell at 0x10c9a6440: dict object at 0x10ca27600>, '__classcell__': <cell at 0x10c9a6770: AMeta object at 0x9950c

In [2]:
steve = Person("Steve")

AMeta __call__ <class '__main__.Person'> ('Steve',) {}
Person __new__ <class '__main__.Person'> Steve
Person __init__ Steve


In [4]:
class Foo:
    name = "foo_name"

    def __init__(self, val):
        self.val = val


ZooNew = AMeta("Zoo", (Foo,), {"age": 99})

AMeta.__new__ name='Zoo', bases=(<class '__main__.Foo'>,), classdict={'age': 99}, kwargs={}
AMeta __new__ <class '__main__.Zoo'>
AMeta __init__ <class '__main__.Zoo'> Zoo (<class '__main__.Foo'>,) {'age': 99}


In [5]:
zoo = ZooNew(10)

AMeta __call__ <class '__main__.Zoo'> (10,) {}


In [6]:
zoo

In [7]:
ZooNew.__name__

'Zoo'

In [8]:
Zoo2 = ZooNew

In [9]:
Zoo2.__name__

'Zoo'

In [10]:
Foo2 = Foo

In [11]:
Foo2.__name__

'Foo'

In [26]:
def wrile_log(self):
    self.__private = "priv_attr"
    print(f"write_log {self=}")


ZooMethods = AMeta(
    "Zoo",
    (Foo,),
    {
        "age": 99,
        "write_log": wrile_log,
        "print": lambda self: setattr(self, "__private_lambda", f"print {self=}"),
    }
)

AMeta.__new__ name='Zoo', bases=(<class '__main__.Foo'>,), classdict={'age': 99, 'write_log': <function wrile_log at 0x10ccbeae0>, 'print': <function <lambda> at 0x10ccbeb90>}, kwargs={}
AMeta __new__ <class '__main__.Zoo'>
AMeta __init__ <class '__main__.Zoo'> Zoo (<class '__main__.Foo'>,) {'age': 99, 'write_log': <function wrile_log at 0x10ccbeae0>, 'print': <function <lambda> at 0x10ccbeb90>}


In [27]:
zoo = ZooMethods(10)

AMeta __call__ <class '__main__.Zoo'> (10,) {}


In [28]:
zoo.print()

In [29]:
zoo.write_log()

write_log self=<__main__.Zoo object at 0x10cc9b4d0>


In [30]:
zoo

In [31]:
zoo.__dict__

{'val': 10,
 '__private_lambda': 'print self=<__main__.Zoo object at 0x10cc9b4d0>',
 '__private': 'priv_attr'}

In [40]:
class Singleton:
    _instance = None

    def __new__(cls, *args, **kwargs):
        print("Singleton.__new__", cls)
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance


class Animal(Singleton):
    def __init__(self, name):
        print(f"Animal.__init__, {name=}")
        self.name = name


class Person(Singleton):
    def __init__(self, name):
        print(f"Person.__init__, {name=}")
        self.name = name

In [43]:
tiger = Animal("tiger")

Singleton.__new__ <class '__main__.Animal'>
Animal.__init__, name='tiger'


In [44]:
tiger.__dict__

{'name': 'tiger'}

In [45]:
Animal.__dict__

mappingproxy({'__module__': '__main__',
              '__firstlineno__': 11,
              '__init__': <function __main__.Animal.__init__(self, name)>,
              '__static_attributes__': ('name',),
              '__doc__': None,
              '_instance': <__main__.Animal at 0x10d220050>})

In [46]:
Person.__dict__

mappingproxy({'__module__': '__main__',
              '__firstlineno__': 17,
              '__init__': <function __main__.Person.__init__(self, name)>,
              '__static_attributes__': ('name',),
              '__doc__': None})

In [47]:
Person._instance

In [48]:
Animal._instance

In [49]:
tiger.name

'tiger'

In [50]:
cat = Animal("cat")

Singleton.__new__ <class '__main__.Animal'>
Animal.__init__, name='cat'


In [51]:
tiger is cat

True

In [52]:
tiger.name, cat.name

('cat', 'cat')

In [61]:
class SingletonMeta(type):
    _instances = {}

    def __call__(cls, *args, **kwargs):
        print(f"SingletonMeta.__call__ {cls=}, {args=}, {kwargs=}")

        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)

        return cls._instances[cls]


class DbConnector(metaclass=SingletonMeta):
    def __init__(self, dbname):
        print(f"DbConnector.__init__ {dbname=}")
        self.dbname = dbname


class MySqlDbConnector(DbConnector):
    pass

In [65]:
sqlite = DbConnector("sqlite3")

SingletonMeta.__call__ cls=<class '__main__.DbConnector'>, args=('sqlite3',), kwargs={}
DbConnector.__init__ dbname='sqlite3'


In [55]:
sqlite

In [56]:
sqlite.dbname

'sqlite3'

In [57]:
DbConnector.__dict__

mappingproxy({'__module__': '__main__',
              '__firstlineno__': 13,
              '__init__': <function __main__.DbConnector.__init__(self, dbname)>,
              '__static_attributes__': ('dbname',),
              '__dict__': <attribute '__dict__' of 'DbConnector' objects>,
              '__weakref__': <attribute '__weakref__' of 'DbConnector' objects>,
              '__doc__': None})

In [58]:
pg = DbConnector("pgsql")

SingletonMeta.__call__ cls=<class '__main__.DbConnector'>, args=('pgsql',), kwargs={}


In [59]:
pg is sqlite

True

In [60]:
SingletonMeta._instances

{__main__.DbConnector: <__main__.DbConnector at 0x10cc9bcb0>}

In [62]:
mysql = MySqlDbConnector("mysql")

SingletonMeta.__call__ cls=<class '__main__.MySqlDbConnector'>, args=('mysql',), kwargs={}
DbConnector.__init__ dbname='mysql'


In [63]:
mysql2 = MySqlDbConnector("mysql2")

SingletonMeta.__call__ cls=<class '__main__.MySqlDbConnector'>, args=('mysql2',), kwargs={}


In [64]:
mysql2 is mysql

True

In [66]:
SingletonMeta._instances

{__main__.MySqlDbConnector: <__main__.MySqlDbConnector at 0x10d220830>,
 __main__.DbConnector: <__main__.DbConnector at 0x10d220980>}

## ABC

In [67]:
from abc import ABC, ABCMeta, abstractmethod

In [68]:
class BasePerson(ABC):
    @abstractmethod
    def eat(self, food):
        print(f"BasePerson.eat {food=}")

    @abstractmethod
    def sleep(self):
        raise NotImplemented

    def study(self):
        print(f"BasePerson.study")
        self.eat("apple")
        self.sleep()

In [69]:
steve = BasePerson()

TypeError: Can't instantiate abstract class BasePerson without an implementation for abstract methods 'eat', 'sleep'

In [73]:
class Student(BasePerson):
    @abstractmethod
    def sleep(self):
        print(f"Student.sleep")

In [74]:
steve = Student()

TypeError: Can't instantiate abstract class Student without an implementation for abstract methods 'eat', 'sleep'

In [75]:
class Student(BasePerson):
    def sleep(self):
        print(f"Student.sleep")

In [76]:
steve = Student()

TypeError: Can't instantiate abstract class Student without an implementation for abstract method 'eat'

In [81]:
class HSEStudent(Student):
    def eat(self, food):
        super().eat(food)
        print(f"HSEStudent.eat {food=}")

In [82]:
steve = HSEStudent()

In [83]:
steve.eat("bread")

BasePerson.eat food='bread'
HSEStudent.eat food='bread'


In [84]:
steve.sleep()

Student.sleep


In [85]:
steve.study()

BasePerson.study
BasePerson.eat food='apple'
HSEStudent.eat food='apple'
Student.sleep


In [86]:
from collections.abc import Hashable

In [87]:
line = "123qwe"

In [89]:
line.__class__.mro()

[str, object]

In [90]:
isinstance(line, Hashable)

True

In [91]:
line.__hash__

<method-wrapper '__hash__' of str object at 0x10d38e700>

In [92]:
isinstance({}, Hashable)

False

In [93]:
isinstance(steve, Hashable)

True

In [94]:
steve.__hash__

<method-wrapper '__hash__' of HSEStudent object at 0x10d220ec0>

In [98]:
getattr(HSEStudent, "eat"), getattr(HSEStudent, "study")

(<function __main__.HSEStudent.eat(self, food)>,
 <function __main__.BasePerson.study(self)>)

In [104]:
class Predictable(metaclass=ABCMeta):

    @abstractmethod
    def predict(self):
        pass

    @classmethod
    def __subclasshook__(cls, C):
        print(f"Predictable __subclasshook__ {cls=}, {C=}")

        name = "predict"
        for class_ in C.__mro__:
            if name in class_.__dict__ and callable(class_.__dict__[name]):
                return True

        return NotImplemented


class TreeModel:
    def predict(self):
        pass


class TrafficLight:
    predict = "green"


tree = TreeModel()
light = TrafficLight()

print("-" * 20)

print(f"{isinstance(tree, Predictable)=}, {issubclass(TreeModel, Predictable)=}")
print(f"{isinstance(light, Predictable)=}, {issubclass(TrafficLight, Predictable)=}")

--------------------
Predictable __subclasshook__ cls=<class '__main__.Predictable'>, C=<class '__main__.TreeModel'>
isinstance(tree, Predictable)=True, issubclass(TreeModel, Predictable)=True
Predictable __subclasshook__ cls=<class '__main__.Predictable'>, C=<class '__main__.TrafficLight'>
isinstance(light, Predictable)=False, issubclass(TrafficLight, Predictable)=False


In [96]:
tree.predict()

In [109]:
class Predictable(metaclass=ABCMeta):

    @abstractmethod
    def predict(self):
        pass

    @classmethod
    def __subclasshook__(cls, C):
        #print(f"Predictable __subclasshook__ {cls=}, {C=}")

        if cls is not Predictable:
            return NotImplemented

        name = "predict"
        for class_ in C.__mro__:
            if name in class_.__dict__ and callable(class_.__dict__[name]):
                return True

        return NotImplemented


class TrainPredictable(Predictable):

    @abstractmethod
    def train(self):
        pass

    @classmethod
    def __subclasshook__(cls, C):
        #print(f"Predictable __subclasshook__ {cls=}, {C=}")

        if cls is not TrainPredictable:
            return NotImplemented

        for name in ("predict", "train"):
            for class_ in C.__mro__:
                if name in class_.__dict__ and callable(class_.__dict__[name]):
                    break
            else:
                return NotImplemented

        return True


class TreeModel:
    def predict(self):
        pass


class RandomForestModel:
    def train(self):
        pass

    def predict(self):
        pass


class TrafficLight:
    predict = "green"


tree = TreeModel()
forest = RandomForestModel()
light = TrafficLight()

print("-" * 20)

print(f"{isinstance(tree, Predictable)=}, {issubclass(TreeModel, Predictable)=}")
print(f"{isinstance(light, Predictable)=}, {issubclass(TrafficLight, Predictable)=}")
print(f"{isinstance(forest, Predictable)=}, {issubclass(RandomForestModel, Predictable)=}")

print("-" * 20)
print(f"{isinstance(tree, TrainPredictable)=}, {issubclass(TreeModel, TrainPredictable)=}")
print(f"{isinstance(forest, TrainPredictable)=}, {issubclass(RandomForestModel, TrainPredictable)=}")

--------------------
isinstance(tree, Predictable)=True, issubclass(TreeModel, Predictable)=True
isinstance(light, Predictable)=False, issubclass(TrafficLight, Predictable)=False
isinstance(forest, Predictable)=True, issubclass(RandomForestModel, Predictable)=True
--------------------
isinstance(tree, TrainPredictable)=False, issubclass(TreeModel, TrainPredictable)=False
isinstance(forest, TrainPredictable)=True, issubclass(RandomForestModel, TrainPredictable)=True


In [113]:
class AttrAccess:
    def __init__(self, name):
        self.__name = name

    @property
    def name(self):
        return self.__name

    def __getattribute__(self, attr):
        print("__getattribute__", attr)
        return super().__getattribute__(attr)

    def __setattr__(self, attr, val):
        print("__setattr__", attr, val)
        return super().__setattr__(attr, val)

In [114]:
steve = AttrAccess("Steve")

__getattribute__ __class__
__getattribute__ __class__
__getattribute__ __class__
__getattribute__ __class__
__setattr__ _AttrAccess__name Steve


In [112]:
steve.name

__getattribute__ name


NameError: name 'self' is not defined

In [115]:
class AttrAccess:
    def __init__(self, name):
        self.__name = name

    @property
    def name(self):
        return self.__name

    def __getattribute__(self, attr):
        print("__getattribute__", attr)
        return super().__getattribute__(attr)

    def __setattr__(self, attr, val):
        print("__setattr__", attr, val)
        return super().__setattr__(attr, val)

steve = AttrAccess("Steve")

__setattr__ _AttrAccess__name Steve


In [116]:
steve.name

__getattribute__ name
__getattribute__ _AttrAccess__name


'Steve'

In [117]:
steve.qwerty

__getattribute__ qwerty


AttributeError: 'AttrAccess' object has no attribute 'qwerty'

In [118]:
steve.name = "123"

__setattr__ name 123


AttributeError: property 'name' of 'AttrAccess' object has no setter

In [119]:
steve.x = "123"

__setattr__ x 123


In [120]:
steve.__dict__

__getattribute__ __dict__


{'_AttrAccess__name': 'Steve', 'x': '123'}

In [127]:
class AttrAccess:
    def __init__(self, name):
        self.__name = name

    @property
    def name(self):
        return self.__name

    def __getattribute__(self, attr):
        print("__getattribute__", attr)
        return super().__getattribute__(attr)

    def __getattr__(self, attr):
        print("__getattr__", attr)

    def __setattr__(self, attr, val):
        print("__setattr__", attr, val)
        return super().__setattr__(attr, val)

steve = AttrAccess("Steve")

__setattr__ _AttrAccess__name Steve


In [128]:
steve.name

__getattribute__ name
__getattribute__ _AttrAccess__name


'Steve'

In [129]:
steve.qwerty

__getattribute__ qwerty
__getattr__ qwerty


In [130]:
class AttrAccess:
    def __init__(self, name):
        self.__name = name

    @property
    def name(self):
        return self.__name

    def __getattribute__(self, attr):
        print("__getattribute__", attr)
        return super().__getattribute__(attr)

    def __getattr__(self, attr):
        print("__getattr__", attr)
        return super().__getattr__(attr)

    def __setattr__(self, attr, val):
        print("__setattr__", attr, val)
        return super().__setattr__(attr, val)

steve = AttrAccess("Steve")

__setattr__ _AttrAccess__name Steve


In [131]:
steve.name

__getattribute__ name
__getattribute__ _AttrAccess__name


'Steve'

In [132]:
steve.qwerty

__getattribute__ qwerty
__getattr__ qwerty


AttributeError: 'super' object has no attribute '__getattr__'

In [133]:
from functools import partial, reduce

In [137]:
reduce(lambda x, y: x * y, [1, 2, 3])

6

In [138]:
def process(url, callback):
    # request.get()
    data = "get_result"
    callback(data)


def collect_data(data, timestamp, user):
    print("collect_data", data, timestamp, user)



part_collect = partial(collect_data, timestamp=123, user="steve")


process("url", part_collect)

collect_data get_result 123 steve


In [139]:
part_collect

functools.partial(<function collect_data at 0x10d7e77f0>, timestamp=123, user='steve')

In [140]:
part_collect = partial(collect_data, timestamp=123, user="steve")

In [141]:
part_collect()

TypeError: collect_data() missing 1 required positional argument: 'data'

In [142]:
part_collect("data")

collect_data data 123 steve
